<a href="https://colab.research.google.com/github/mxls34/AdvanceDatabase/blob/main/Chapter4_Subqueries_CTE_QueryRewriting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 4: Subqueries, CTE & Query Rewriting

หัวข้อในบทนี้
1. Subqueries (scalar ใน WHERE/SELECT/FROM, correlated, IN, ANY/ALL/SOME)
2. Semi-Join & Anti-Join (EXISTS, NOT EXISTS, กับดัก NOT IN + NULL)
3. CTE & Recursive CTE
4. Query Rewriting & Anti-Patterns

## เตรียมฐานข้อมูล

ติดตั้ง DuckDB และสร้างฐานข้อมูลจำลอง (ห้องสมุด: `member`, `book`, `loan`)

In [ ]:
!pip install duckdb --quiet

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')

def run(sql: str) -> pd.DataFrame:
    """รันคำสั่ง SQL แล้วคืนผลลัพธ์เป็น pandas DataFrame"""
    return con.sql(sql).df()

print("พร้อมใช้งาน DuckDB เวอร์ชัน:", duckdb.__version__)

พร้อมใช้งาน DuckDB เวอร์ชัน: 1.3.2


In [ ]:
%%time
con.execute("""
CREATE OR REPLACE TABLE member (
    member_id   INTEGER PRIMARY KEY,
    name        VARCHAR,
    city        VARCHAR,
    referred_by INTEGER
);
INSERT INTO member VALUES
    (1, 'สมชาย', 'กรุงเทพ',    NULL),
    (2, 'สุดา',   'กรุงเทพ',    1),
    (3, 'วิชัย',  'เชียงใหม่',  NULL),
    (4, 'มานี',   'ขอนแก่น',    NULL),
    (5, 'ปิติ',   'กรุงเทพ',    2),
    (6, 'ก้อง',   'เชียงใหม่',  3),
    (7, 'แก้ว',   'เชียงใหม่',  6);

CREATE OR REPLACE TABLE book (
    book_id  INTEGER PRIMARY KEY,
    title    VARCHAR,
    category VARCHAR,
    price    INTEGER
);
INSERT INTO book VALUES
    (1, 'ฐานข้อมูลเบื้องต้น',         'เทคโนโลยี',    420),
    (2, 'วิทยาการข้อมูล',            'เทคโนโลยี',    380),
    (3, 'ปัญญาประดิษฐ์เบื้องต้น',     'เทคโนโลยี',    450),
    (4, 'ประวัติศาสตร์ไทย',          'ประวัติศาสตร์', 300),
    (5, 'สงครามโลกครั้งที่สอง',      'ประวัติศาสตร์', 320),
    (6, 'นิยายรักฤดูฝน',             'นิยาย',        250),
    (7, 'ตำนานเมืองเหนือ',           'นิยาย',        420),
    (8, 'บันทึกนักเดินทาง',          'นิยาย',        280);

CREATE OR REPLACE TABLE loan (
    loan_id   INTEGER PRIMARY KEY,
    member_id INTEGER,
    book_id   INTEGER
);
INSERT INTO loan VALUES
    (1, 1, 1),   -- สมชาย ยืม ฐานข้อมูลเบื้องต้น (เทคโนโลยี)
    (2, 1, 3),   -- สมชาย ยืม ปัญญาประดิษฐ์เบื้องต้น (เทคโนโลยี)
    (3, 2, 2),   -- สุดา ยืม วิทยาการข้อมูล (เทคโนโลยี)
    (4, 2, 6),   -- สุดา ยืม นิยายรักฤดูฝน (นิยาย)
    (5, 3, 1);   -- วิชัย ยืม ฐานข้อมูลเบื้องต้น (เทคโนโลยี)
""")
print("สร้างตาราง member, book, loan เรียบร้อย")

สร้างตาราง member, book, loan เรียบร้อย
CPU times: user 17.6 ms, sys: 1.47 ms, total: 19 ms
Wall time: 35.6 ms


In [ ]:
# เขียนคำสั่งดึงข้อมูลในตาราง member ออกมาแสดง
run("....")

,member_id,name,city,referred_by
0,1,สมชาย,กรุงเทพ,<NA>
1,2,สุดา,กรุงเทพ,1
2,3,วิชัย,เชียงใหม่,<NA>
3,4,มานี,ขอนแก่น,<NA>
4,5,ปิติ,กรุงเทพ,2
5,6,ก้อง,เชียงใหม่,3
6,7,แก้ว,เชียงใหม่,6


In [ ]:
# เขียนคำสั่งดึงข้อมูลในตาราง book ออกมาแสดง
run("....")

,book_id,title,category,price
0,1,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420
1,2,วิทยาการข้อมูล,เทคโนโลยี,380
2,3,ปัญญาประดิษฐ์เบื้องต้น,เทคโนโลยี,450
3,4,ประวัติศาสตร์ไทย,ประวัติศาสตร์,300
4,5,สงครามโลกครั้งที่สอง,ประวัติศาสตร์,320
5,6,นิยายรักฤดูฝน,นิยาย,250
6,7,ตำนานเมืองเหนือ,นิยาย,420
7,8,บันทึกนักเดินทาง,นิยาย,280


In [ ]:
# เขียนคำสั่งดึงข้อมูลในตาราง loan ออกมาแสดง
run("....")

,loan_id,member_id,book_id
0,1,1,1
1,2,1,3
2,3,2,2
3,4,2,6
4,5,3,1


**สรุปข้อมูล**: สมาชิก 7 คน — **สมชาย, สุดา, วิชัย** เคยยืมหนังสือ (สมชาย 2 เล่ม, สุดา 2 เล่ม, วิชัย 1 เล่ม)

ส่วน **มานี, ปิติ, ก้อง, แก้ว** ไม่เคยยืมเลย หนังสือมี 3 หมวด: เทคโนโลยี, ประวัติศาสตร์, นิยาย

## 1. Subqueries

**Subquery** คือ query ที่ซ้อนอยู่ภายใน query อีกอันหนึ่ง

วางได้ 3 ตำแหน่งหลัก
* ใน `WHERE` (เป็นเงื่อนไข),
* ใน `SELECT` (เป็นคอลัมน์เพิ่ม),
* และใน `FROM` (เป็น derived table)

### 1.1 Scalar Subquery ใน WHERE

**Scalar subquery** คือ subquery ที่คืนค่าเดียว (1 แถว 1 คอลัมน์) ใช้เป็นเงื่อนไขเปรียบเทียบได้เลย

**ตัวอย่าง**: หาหนังสือที่ราคาสูงกว่าค่าเฉลี่ย

In [ ]:
run("""
SELECT title, price
FROM book
WHERE price > (SELECT AVG(price) FROM book)
ORDER BY price DESC;
""")

,title,price
0,ปัญญาประดิษฐ์เบื้องต้น,450
1,ฐานข้อมูลเบื้องต้น,420
2,ตำนานเมืองเหนือ,420
3,วิทยาการข้อมูล,380


**แบบฝึกหัด 1.1**

หาหนังสือที่ราคา**ต่ำกว่า**ค่าเฉลี่ย เรียงจากราคาน้อยไปมาก

In [ ]:
# เขียนโค้ดข้อ 1.1
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.1

,title,price
0,นิยายรักฤดูฝน,250
1,บันทึกนักเดินทาง,280
2,ประวัติศาสตร์ไทย,300
3,สงครามโลกครั้งที่สอง,320


### 1.2 Scalar Subquery ใน SELECT

ใส่ scalar subquery ไว้ตรงตำแหน่งคอลัมน์ จะได้คอลัมน์ใหม่ที่มีค่า**เดียวกันซ้ำทุกแถว**

**ตัวอย่าง**: หาราคาเป็น % ของค่าเฉลี่ย

In [ ]:
run("""
SELECT title, price,
  ROUND(price * 100.0 / (SELECT AVG(price) FROM book), 1) AS pct_of_avg
FROM book
ORDER BY price DESC;
""")

,title,price,pct_of_avg
0,ปัญญาประดิษฐ์เบื้องต้น,450,127.7
1,ฐานข้อมูลเบื้องต้น,420,119.1
2,ตำนานเมืองเหนือ,420,119.1
3,วิทยาการข้อมูล,380,107.8
4,สงครามโลกครั้งที่สอง,320,90.8
5,ประวัติศาสตร์ไทย,300,85.1
6,บันทึกนักเดินทาง,280,79.4
7,นิยายรักฤดูฝน,250,70.9


**แบบฝึกหัด 1.2**

หาผลต่างของราคากับค่าเฉลี่ย กำหนดให้ชื่อคอลัมน์เป็น `diff_from_avg`

In [ ]:
# เขียนโค้ดข้อ 1.2
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.2

,title,price,diff_from_avg
0,ปัญญาประดิษฐ์เบื้องต้น,450,97.5
1,ฐานข้อมูลเบื้องต้น,420,67.5
2,ตำนานเมืองเหนือ,420,67.5
3,วิทยาการข้อมูล,380,27.5
4,สงครามโลกครั้งที่สอง,320,-32.5
5,ประวัติศาสตร์ไทย,300,-52.5
6,บันทึกนักเดินทาง,280,-72.5
7,นิยายรักฤดูฝน,250,-102.5


### 1.3 Scalar Subquery ใน FROM (Derived Table) — aggregate ซ้อน aggregate

ในการเขียน SQL สำหรับ **aggregate สองชั้น** เช่น "แต่ละเมือง มียอดยืมหนังสือเฉลี่ยต่อคนเท่าไหร่" — ต้องนับต่อคนก่อน (ชั้นใน) แล้วค่อยเฉลี่ยตามเมือง (ชั้นนอก) เขียน `GROUP BY` ชั้นเดียวไม่ได้ ต้องห่อ query ชั้นในด้วย derived table (ต้องตั้ง alias เสมอ)

**ตัวอย่าง** ให้หาว่าแต่ละเมือง มียอดยืมหนังสือเฉลี่ยต่อคนเท่าไหร่

In [ ]:
run("""
SELECT city, ROUND(AVG(cnt), 2) AS avg_per_member
FROM (
  SELECT m.member_id, m.city, COUNT(l.loan_id) AS cnt
  FROM member m
  LEFT JOIN loan l ON m.member_id = l.member_id
  GROUP BY m.member_id, m.city
) t
GROUP BY city
ORDER BY avg_per_member DESC;
""")

,city,avg_per_member
0,กรุงเทพ,1.33
1,เชียงใหม่,0.33
2,ขอนแก่น,0.00


**แบบฝึกหัด 1.3**

ต้องการหาราคาเฉลี่ยของหนังสือที่ถูกยืมในแต่ละเมือง — ต้องรวมราคาหนังสือที่แต่ละคนยืมก่อน (ชั้นใน) แล้วค่อยเฉลี่ยตามเมือง (ชั้นนอก)

In [ ]:
# เขียนโค้ดข้อ 1.3
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.3

,city,avg_spent_per_member
0,กรุงเทพ,500.0
1,เชียงใหม่,140.0
2,ขอนแก่น,0.0


### 1.4 Correlated Subquery

**Correlated subquery** คือ subquery ที่อ้างอิงถึงคอลัมน์ของ query ชั้นนอก จึงรันแยกเดี่ยวๆ ไม่ได้ และต้องรันซ้ำใหม่**ทุกแถว**ของ query ชั้นนอก (ช้ากว่า non-correlated subquery)

**ตัวอย่างที่ 1 (ใน SELECT)**

หาจำนวนการยืมของแต่ละคน

In [ ]:
run("""
SELECT m.name,
  (SELECT COUNT(*) FROM loan l WHERE l.member_id = m.member_id) AS loans
FROM member m
ORDER BY loans DESC;
""")

,name,loans
0,สมชาย,2
1,สุดา,2
2,วิชัย,1
3,มานี,0
4,ปิติ,0
5,ก้อง,0
6,แก้ว,0


**ตัวอย่างที่ 2 (ใน WHERE)**

หาหนังสือที่แพงสุดในแต่ละหมวด

In [ ]:
run("""
SELECT title, category, price
FROM book b
WHERE price = (
  SELECT MAX(price) FROM book b2
  WHERE b2.category = b.category
);
""")

,title,category,price
0,ปัญญาประดิษฐ์เบื้องต้น,เทคโนโลยี,450
1,สงครามโลกครั้งที่สอง,ประวัติศาสตร์,320
2,ตำนานเมืองเหนือ,นิยาย,420




**แบบฝึกหัด 1.4**

หาหนังสือที่**ราคาถูกสุด**ในแต่ละหมวด

In [ ]:
# เขียนโค้ดข้อ 1.4
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.4

,title,category,price
0,นิยายรักฤดูฝน,นิยาย,250
1,ประวัติศาสตร์ไทย,ประวัติศาสตร์,300
2,วิทยาการข้อมูล,เทคโนโลยี,380


### 1.5 IN Subquery

`IN` ร่วมกับ subquery เช็คว่าค่าหนึ่งๆ**อยู่ในชุดค่า**ที่ subquery คืนมาหรือไม่ (subquery ต้องคืน **1 คอลัมน์เท่านั้น** แต่กี่แถวก็ได้)

**ตัวอย่าง**: หาสมาชิกที่เคยยืมหนังสือหมวดเทคโนโลยี

In [ ]:
run("""
SELECT name FROM member
WHERE member_id IN (
  SELECT l.member_id FROM loan l
  JOIN book b ON l.book_id = b.book_id
  WHERE b.category = 'เทคโนโลยี'
);
""")

,name
0,สมชาย
1,สุดา
2,วิชัย


**แบบฝึกหัด 1.5**

หาสมาชิกที่เคยยืมหนังสือหมวด**นิยาย**

In [ ]:
# เขียนโค้ดข้อ 1.5
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.5

,name
0,สุดา


### 1.6 ANY / ALL / SOME

ใช้คู่กับ operator เปรียบเทียบ (`>`, `<`, `=` ฯลฯ) เพื่อเทียบค่าหนึ่งกับ**หลายค่า**ที่ subquery คืนมา

| ตัวดำเนินการ | ความหมาย | เทียบเท่า |
|---|---|---|
| `ALL` | ต้องเป็นจริงกับ**ทุกค่า** | `> ALL(...)` ≈ `> MAX(...)` |
| `ANY` / `SOME` | เป็นจริงแค่**ค่าใดค่าหนึ่ง**ก็พอ | `> ANY(...)` ≈ `> MIN(...)` |

**ตัวอย่าง**: ต้องการหาว่ามีหนังสือทุกเล่ม ที่แพงกว่าอย่างน้อยหนึ่งเล่มในหมวดนิยาย

In [ ]:
run("""
SELECT title, price FROM book
WHERE price > ALL (
  SELECT price FROM book WHERE category = 'นิยาย'
);
""")

,title,price
0,ปัญญาประดิษฐ์เบื้องต้น,450


**แบบฝึกหัด 1.6**

หาหนังสือที่ราคา**ถูกกว่าทุกเล่ม**ในหมวดเทคโนโลยี

In [ ]:
# เขียนโค้ดข้อ 1.6

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.6

,title,price
0,ประวัติศาสตร์ไทย,300
1,สงครามโลกครั้งที่สอง,320
2,นิยายรักฤดูฝน,250
3,บันทึกนักเดินทาง,280


## 2. Semi-Join & Anti-Join

หาแถวโดยดูว่า "เจอคู่ในอีกตารางไหม" โดย**ไม่ต้องดึงคอลัมน์**จากตารางนั้นมาแสดง (ต่างจาก JOIN ทั่วไป) และไม่เกิดแถวซ้ำเหมือน JOIN

- **Semi-Join** เขียนด้วย `EXISTS` — หาแถวที่ "มีคู่"
- **Anti-Join** เขียนด้วย `NOT EXISTS` — หาแถวที่ "ไม่มีคู่เลย"

**ตัวอย่าง**

หาสมาชิกที่เคยยืม (semi-join)

In [ ]:
run("""
SELECT m.name FROM member m
WHERE EXISTS (
  SELECT 1 FROM loan l WHERE l.member_id = m.member_id
);
""")

,name
0,สมชาย
1,สุดา
2,วิชัย


หาสมาชิกที่ไม่เคยยืม (anti-join)

In [ ]:
run("""
SELECT m.name FROM member m
WHERE NOT EXISTS (
  SELECT 1 FROM loan l WHERE l.member_id = m.member_id
);
""")

,name
0,มานี
1,ปิติ
2,ก้อง
3,แก้ว


**แบบฝึกหัด 2.1**

หาสมาชิกที่**ไม่เคย**ยืมหนังสือหมวด **"ประวัติศาสตร์"** โดยใช้ `NOT EXISTS`

In [ ]:
# เขียนโค้ดข้อ 2.1
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 2.1

,name
0,ปิติ
1,สุดา
2,แก้ว
3,สมชาย
4,มานี
5,วิชัย
6,ก้อง


## 3. CTE & Recursive CTE

**CTE (Common Table Expression)** คือการตั้งชื่อให้ subquery ล่วงหน้าด้วย `WITH` แล้วเรียกใช้เหมือนตารางจริงในคำสั่งหลัก — ช่วยให้อ่านง่ายกว่า derived table

### 3.1 CTE พื้นฐาน

In [ ]:
run("""
WITH loan_count AS (
  SELECT member_id, COUNT(*) AS l_count FROM loan GROUP BY member_id
)
SELECT m.name, lc.l_count
FROM member m JOIN loan_count lc ON m.member_id = lc.member_id
ORDER BY lc.l_count DESC;
""")

,name,l_count
0,สมชาย,2
1,สุดา,2
2,วิชัย,1


**แบบฝึกหัด 3.1**

ใช้ CTE ก่อนหน้า ในการหาสมาชิกที่ยืม**มากกว่า 1 ครั้ง**

In [ ]:
# เขียนโค้ดข้อ 3.1
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์

,name,c
0,สมชาย,2
1,สุดา,2


### 3.2 CTE หลายตัวต่อกัน

CTE สามารถเขียนหลายตัวซ้อนกันได้ ให้ตัวหลังอ้างอิงตัวก่อนหน้า — ทำให้โจทย์ใหญ่ๆ แตกเป็นขั้นตอนย่อยที่อ่านได้ง่ายขึ้นและเป็นลำดับ

**ตัวอย่าง**

ให้หาว่า "สมาชิกแต่ละคน ยืมหนังสือหมวดเทคโนโลยีไปกี่เล่ม" โดยให้ แสดงเฉพาะคนที่เคยยืมหมวดนี้อย่างน้อย 1 เล่ม พร้อมจำนวนครั้งที่ยืม

In [ ]:
run("""
WITH tech AS (
  SELECT book_id FROM book WHERE category = 'เทคโนโลยี'
),
tech_loans AS (
  SELECT member_id, COUNT(*) AS l_count FROM loan
  WHERE book_id IN (SELECT book_id FROM tech)
  GROUP BY member_id
)
SELECT m.name, tl.l_count FROM member m
JOIN tech_loans tl ON m.member_id = tl.member_id;
""")

,name,l_count
0,สมชาย,2
1,สุดา,1
2,วิชัย,1


**แบบฝึกหัด 3.2**

ให้หาว่า "สมาชิกแต่ละคน ยืมหนังสือหมวดนิยายไปกี่เล่ม"

In [ ]:
# เขียนโค้ดข้อ 3.2
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 3.2

,name,category,l_count
0,วิชัย,เทคโนโลยี,1
1,สมชาย,เทคโนโลยี,2
2,สุดา,นิยาย,1
3,สุดา,เทคโนโลยี,1


### 3.3 Recursive CTE

ใช้กับข้อมูลแบบลำดับชั้น (hierarchical) ที่ไม่รู้ล่วงหน้าว่าลึกกี่ระดับ ประกอบด้วย 2 ส่วนคั่นด้วย `UNION ALL`
- **Anchor member**: จุดเริ่มต้น รันครั้งเดียว ไม่อ้างอิงตัวเอง
- **Recursive member**: อ้างอิงชื่อ CTE ของตัวเองซ้ำ เพื่อหา "ชั้นถัดไป"

**ตัวอย่าง**: สายการชักชวนสมาชิก (`referred_by`)

In [ ]:
run("""
WITH RECURSIVE chain AS (
  -- anchor: เริ่มจากคนที่ไม่มีใครชวน

  SELECT member_id, name, referred_by, 1 AS depth
  FROM member WHERE referred_by IS NULL

  UNION ALL

  -- recursive: คนที่ถูกชวนโดยคนใน chain รอบก่อน

  SELECT m.member_id, m.name, m.referred_by, c.depth + 1
  FROM member m
  JOIN chain c ON m.referred_by = c.member_id
)

SELECT
  chain.name,
  ref.name AS referred_by_name,
  chain.depth
FROM chain
LEFT JOIN member ref ON chain.referred_by = ref.member_id
ORDER BY chain.depth;
""")

,name,referred_by_name,depth
0,สมชาย,None,1
1,วิชัย,None,1
2,มานี,None,1
3,สุดา,สมชาย,2
4,ก้อง,วิชัย,2
5,ปิติ,สุดา,3
6,แก้ว,ก้อง,3


**แบบฝึกหัด 3.3**:

จาก CTE ก่อนหน้า ให้หาว่าใครที่ถูกชักชวนตั้งแต่ลำดับ (depth) ที่ 3 เป็นต้นไป

In [ ]:
# เขียนโค้ดข้อ 3.3
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 3.3

,name,depth
0,ปิติ,3
1,แก้ว,3


### 3.4 ข้อควรระวังสำหรับการเขียน Recursive CTE ที่อาจจะวนไม่รู้จบ (Cycle)

ถ้าข้อมูลอ้างอิงกันเป็น**วงกลม** (เช่น 1→3→2→1) recursive CTE จะวนซ้ำไม่มีวันจบ เพราะไม่มีรอบไหนที่ "ไม่มีแถวใหม่" เกิดขึ้นเอง

> ตัวอย่างเช่น
>
> ```sql
> -- referred_by วนกลับ: 1 -> 3 -> 2 -> 1 (loop) — ❌ ไม่มีเงื่อนไขหยุด
> WITH RECURSIVE chain AS (
>   SELECT id, 1 AS depth FROM member_cycle_demo WHERE id = 1
>   UNION ALL
>   SELECT m.id, c.depth + 1
>   FROM member_cycle_demo m JOIN chain c ON m.ref = c.id
> )
> SELECT * FROM chain;   -- จะรันไม่หยุด!
> ```

**วิธีป้องกัน**: ใส่ระดับความลึกป้องกันไว้เสมอ เช่น `WHERE depth < N`

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE member_cycle_demo (id INTEGER, ref INTEGER);
INSERT INTO member_cycle_demo VALUES (1, 3), (2, 1), (3, 2);  -- 1 -> 3 -> 2 -> 1 วนเป็นวงกลม
""")

# เพิ่ม WHERE c.depth < 10 สำหรับป้องกันวน Loop ไม่รู้จบ
run("""
WITH RECURSIVE chain AS (
  SELECT id, 1 AS depth FROM member_cycle_demo WHERE id = 1
  UNION ALL
  SELECT m.id, c.depth + 1
  FROM member_cycle_demo m JOIN chain c ON m.ref = c.id
  WHERE c.depth < 10
)
SELECT * FROM chain;
""")

,id,depth
0,1,1
1,2,2
2,3,3
3,1,4
4,2,5
5,3,6
6,1,7
7,2,8
8,3,9
9,1,10


สังเกตว่าค่า `id` วนซ้ำ (1, 3, 2, 1, 3, 2, ...) ไปเรื่อยๆ ตาม depth ที่เพิ่มขึ้น — ถ้าไม่มี `WHERE c.depth < 10` คำสั่งนี้จะรันไม่หยุดจริงๆ นี่คือเหตุผลที่ **ควรใส่ตัวจำกัดความลึกทุกครั้ง** ที่เขียน recursive CTE กับข้อมูลที่ไม่มั่นใจว่าเป็นวงกลมหรือไม่

### 3.5 เลือกใช้ CTE หรือ View?

| | CTE (`WITH`) | View |
|---|---|---|
| การใช้งาน | ชั่วคราว ใช้ได้แค่ query เดียว | ค้างไว้ถาวรในฐานข้อมูล |
| ใช้ซ้ำข้าม query | ทำไม่ได้ | ได้ |
| เหมาะกับ | จัดโครงคำสั่งให้อ่านง่าย หรือ recursive | ตรรกะที่ใช้บ่อย/หลายคนต้องใช้ร่วมกัน |

**หลักการเลือก**: ใช้ครั้งเดียว → `CTE` | ใช้ซ้ำหลาย query/หลายคนเรียกใช้ → `View`

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW loan_count_view AS
  SELECT member_id, COUNT(*) AS c FROM loan GROUP BY member_id;
""")

# เรียกใช้ซ้ำได้จากหลาย query โดยไม่ต้องเขียน logic ใหม่
run("SELECT * FROM loan_count_view WHERE c > 1;")

,member_id,c
0,1,2
1,2,2


In [ ]:
run("SELECT AVG(c) AS avg_loans FROM loan_count_view;")

**แบบฝึกหัด 3.5**

ถ้าทีมของคุณต้องดึง "ยอดยืมต่อคน" ไปใช้ในรายงาน 5 รายงานที่ต่างกัน ควรใช้ CTE หรือ View? เพราะอะไร?

<details>
<summary>คำตอบ</summary>

ควรใช้ **View** เพราะ "ยอดยืมต่อคน" ต้องถูกเรียกใช้**ซ้ำในหลาย query** (เช่น จาก 5 รายงาน) — ถ้าใช้ CTE จะต้องคัดลอกคำสั่ง `WITH loan_count AS (...)` ไปวางซ้ำทุกรายงาน แก้ไขทีต้องแก้หลายที่ ในขณะที่ View นิยามครั้งเดียว ทุกรายงานเรียก `FROM loan_count_view` ได้เลย และถ้าชุดคำสั่งเปลี่ยน ก็แก้ที่ View จุดเดียว
</details>

คำตอบคือ ....

## 4. Query Rewriting & Anti-Patterns

SQL มักมีหลายวิธีที่เขียนแล้วได้ผลลัพธ์**เหมือนกัน** แต่**ประสิทธิภาพต่างกัน** — **Query Rewriting** คือทักษะการมองว่า query แบบไหน "ช้าโดยไม่จำเป็น" แล้วเขียนใหม่ให้เร็วขึ้นหรืออ่านง่ายขึ้น โดยไม่เปลี่ยนผลลัพธ์

**Anti-Pattern** คือรูปแบบที่ "ดูเหมือนใช้ได้ แต่จริงๆ เป็นวิธีที่ไม่ดี" — รันผ่าน ได้ผลถูกต้อง แต่มีปัญหาแฝงอยู่ (เช่น ช้า/เสี่ยง NULL)

### 4.1 Correlated Subquery ที่ช้าโดยไม่จำเป็น เทียบกับ JOIN + HAVING

โจทย์:

หาสมาชิกที่ยืมหนังสืออย่างน้อย 2 ครั้ง

แบบ correlated (รัน subquery ซ้ำทุกแถวของ member)

In [ ]:
run("""
SELECT m.name FROM member m
WHERE (SELECT COUNT(*) FROM loan l
       WHERE l.member_id = m.member_id) >= 2;
""")

,name
0,สมชาย
1,สุดา


แบบ JOIN + GROUP BY + HAVING (รวมกลุ่มครั้งเดียว มักเร็วกว่าเมื่อข้อมูลใหญ่)

In [ ]:
run("""
SELECT m.name FROM member m
JOIN loan l ON m.member_id = l.member_id
GROUP BY m.name
HAVING COUNT(*) >= 2;
""")

,name
0,สมชาย
1,สุดา


**แบบฝึกหัด 4.1**

เขียนคำสั่งหา **"สมาชิกที่เคยยืมหนังสือหมวดเทคโนโลยีอย่างน้อย 1 เล่ม"** ด้วย **3 วิธี** ใช้ `IN`, `EXISTS`, และ `JOIN + DISTINCT`

In [ ]:
# เขียนโค้ดข้อ 4.1 ด้วยคำสั่ง IN
run("""

""")

In [ ]:
# เขียนโค้ดข้อ 4.1 ด้วยคำสั่ง EXISTS
run("""

""")

In [ ]:
# เขียนโค้ดข้อ 4.1 ด้วยคำสั่ง JOIN + DISTINCT
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ (ทั้งสามวิธีได้ผลลัพธ์เหมือนกัน)

,name
0,สมชาย
1,สุดา
2,วิชัย


เฉลย

## 5. แบบฝึกหัดท้ายบท

**โจทย์**

หาสมาชิกที่**ไม่เคย**ยืมหนังสือหมวด **"นิยาย"** เลย โดยให้เขียนใน **3 วิธี** แล้วเทียบผลลัพธ์ว่าตรงกันหรือไม่ และอธิบายว่าวิธีไหนควรเลี่ยงไม่ใช้

1. `LEFT JOIN ... WHERE ... IS NULL`
2. `NOT EXISTS`
3. `NOT IN`


In [ ]:
# เขียนโค้ดแบบฝึกหัดท้ายบท ข้อ 1. LEFT JOIN ... WHERE ... IS NULL
run("""

""")

In [ ]:
# เขียนโค้ดแบบฝึกหัดท้ายบท ข้อ 2. NOT EXISTS
run("""

""")

In [ ]:
# เขียนโค้ดแบบฝึกหัดท้ายบท ข้อ 3. NOT IN
run("""

""")

ผลลัพธ์ได้ตรงกันไหมทั้งสามวิธี ....

In [ ]:
# ตัวอย่างผลลัพธ์

,name
0,สมชาย
1,วิชัย
2,มานี
3,ปิติ
4,ก้อง
5,แก้ว


---
## จบบทที่ 4

สรุปสิ่งที่เรียนไปในบทนี้
- **Subquery** วางได้ 3 ตำแหน่ง ได้แก่ scalar (WHERE/SELECT), derived table (FROM), correlated
- **IN / ANY / ALL / SOME** เทียบค่ากับชุดข้อมูลจาก subquery
- **Semi-Join / Anti-Join** (`EXISTS` / `NOT EXISTS`) หาแถวที่มี/ไม่มีคู่ — ปลอดภัยกับ NULL กว่า `IN`/`NOT IN`
- **CTE** (`WITH`) จัดโครง query ให้อ่านง่าย, **Recursive CTE** ไล่ข้อมูลลำดับชั้น (ต้องระวัง cycle ที่อาจจะเกิดขึ้นด้วย)
- **Query Rewriting** query เดียวกันเขียนได้หลายวิธี ควรเลือกวิธีที่เร็ว/อ่านง่ายกว่าโดยไม่เปลี่ยนผลลัพธ์